# 02805 Project: Food.com recipe network

In [ ]:
import os
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")

recipes = pd.read_csv(os.path.join(path, "RAW_recipes.csv"))
interactions = pd.read_csv(os.path.join(path, "RAW_interactions.csv"))
print(recipes.shape, interactions.shape)

## Step 1: Look at the raw data

Before we change anything, we look at what the data contains and list the problems. Each cleaning step later should fix one of these problems.

### Recipes

In [ ]:
recipes.info()

In [ ]:
recipes.isna().sum()

Numeric columns: are there impossible values?

In [ ]:
recipes[["minutes", "n_steps", "n_ingredients"]].describe()

The ingredients look like lists, but are they?

In [ ]:
print(recipes["ingredients"].iloc[0])
print(type(recipes["ingredients"].iloc[0]))

### Reviews (interactions)

In [ ]:
interactions.info()

In [ ]:
interactions["rating"].value_counts().sort_index()

In [ ]:
print("Empty reviews:", interactions["review"].isna().sum())
print("Same user reviewing the same recipe twice:", interactions.duplicated(["user_id", "recipe_id"]).sum())

### Problems found

1. `ingredients` (and `tags`, `steps`, `nutrition`) are stored as text, not as lists.
2. `minutes` has impossible values: minimum 0, maximum about 2 billion.
3. Some recipes have 0 steps or only 1 ingredient.
4. 1 recipe has no name, and 4,979 have no description (not a problem for us).
5. `rating` = 0 occurs 60,847 times. On Food.com this means *no rating given*, not zero stars, so it must be treated as missing.
6. Ratings are very skewed: 816,364 of 1,132,367 (72 %) are 5 stars.
7. 169 reviews are empty. No user reviewed the same recipe twice.

## Step 2: Text columns to lists

`ingredients`, `tags`, `steps` and `nutrition` look like lists but are stored as text, e.g. `"['egg', 'flour']"`. `ast.literal_eval` reads the text as Python and turns it into a real list.

In [ ]:
import ast

for col in ["ingredients", "tags", "steps", "nutrition"]:
    recipes[col] = recipes[col].apply(ast.literal_eval)

print(recipes["ingredients"].iloc[0])
print(type(recipes["ingredients"].iloc[0]))

Check: the length of each new list should match the counts that came with the data.

In [ ]:
print("ingredients match n_ingredients:", (recipes["ingredients"].str.len() == recipes["n_ingredients"]).all())
print("steps match n_steps:", (recipes["steps"].str.len() == recipes["n_steps"]).all())

## Step 3: Impossible cooking times and broken recipes

### Cooking time

First we look at the extreme values before deciding what is wrong.

In [ ]:
m = recipes["minutes"]
print("0 minutes:", (m == 0).sum())
print("More than 1 day:", (m > 24 * 60).sum())
print("More than 1 year:", (m > 365 * 24 * 60).sum())

recipes.loc[m > 24 * 60, ["name", "minutes"]].sample(8, random_state=0)

In [ ]:
recipes.loc[m == 0, ["name", "minutes", "n_steps"]].sample(5, random_state=0)

Recipes over one day are mostly real: marinating, pickling, curing (e.g. sauerbraten takes 4 days). So we keep them. Only two kinds of values are clearly wrong:

- **0 minutes**: no real recipe takes 0 minutes.
- **More than one year**: e.g. 2,147,483,647 minutes, which is the largest number a 32-bit integer can hold, so it is a technical error.

We set these to missing (`NaN`) instead of removing the recipes, because their ingredients are still fine.

In [ ]:
bad = (recipes["minutes"] == 0) | (recipes["minutes"] > 365 * 24 * 60)
recipes.loc[bad, "minutes"] = None
print("Cooking times set to missing:", bad.sum())

### Recipes with no steps or very few ingredients

In [ ]:
print("Recipes with 0 steps:", (recipes["n_steps"] == 0).sum())
print(recipes["n_ingredients"].value_counts().sort_index().head(3))

recipes.loc[recipes["n_ingredients"] <= 2, ["name", "ingredients"]].sample(8, random_state=0)

Only one recipe has no steps at all, so it is not a real recipe and we remove it.

Recipes with 1 or 2 ingredients are real dishes (garlic oil, mango shake), not errors, so we keep them here. Whether they can get links is decided later, when we define the network.

In [ ]:
n_before = len(recipes)
recipes = recipes[recipes["n_steps"] > 0].reset_index(drop=True)
print(f"Removed {n_before - len(recipes)} recipe, {len(recipes):,} left")

## Step 4: Clean the reviews

- **Rating 0**: on Food.com a review can be written without giving stars, and then the rating is stored as 0. So 0 means *no rating*, not *zero stars*, and we set it to missing. We keep the review text.
- **Empty reviews**: a review without text gives us nothing to analyse, so we remove it.
- **Unknown recipes**: reviews of recipes that are no longer in `recipes` (e.g. the one we removed in step 3) are removed.

In [ ]:
import numpy as np

interactions["rating"] = interactions["rating"].replace(0, np.nan)
print("Ratings set to missing:", interactions["rating"].isna().sum())

In [ ]:
n_before = len(interactions)
interactions = interactions[interactions["review"].notna()]
interactions = interactions[interactions["recipe_id"].isin(recipes["id"])]
interactions = interactions.reset_index(drop=True)
print(f"Removed {n_before - len(interactions):,} reviews, {len(interactions):,} left")

Check: the rating distribution without the zeros.

In [ ]:
interactions["rating"].value_counts(normalize=True).sort_index().round(3)

## Step 5: Remove duplicates

We look for three kinds of duplicates:

1. The same recipe id twice.
2. Recipes with exactly the same ingredients, often the same recipe uploaded again under a new name.
3. Duplicate reviews.

In [ ]:
ingr_set = recipes["ingredients"].apply(lambda xs: tuple(sorted(set(xs))))

print("Same id:", recipes["id"].duplicated().sum())
print("Same name and same ingredients:", pd.DataFrame({"n": recipes["name"], "i": ingr_set}).duplicated().sum())
print("Same ingredients:", ingr_set.duplicated().sum())

Examples of recipes with the same ingredients:

In [ ]:
same = recipes[ingr_set.duplicated(keep=False)].assign(ingr_set=ingr_set)
same.sort_values("ingr_set")[["name", "ingredients"]].head(10)

Most of these are clearly the same recipe (e.g. *juicy hamburgers* and *weight watchers juicy hamburgers* with the same 7 ingredients). In the network they would get a link with similarity 1 that only reflects a re-upload, and their reviews are split between the copies.

We keep **one** recipe per ingredient list: the one with the most reviews, because the number of reviews decides which recipes become nodes in the next step. The reviews of the removed copies are removed too.

Note: here two lists only count as the same if the ingredient names are written exactly the same. When we clean the ingredient names later (`eggs` → `egg`), more duplicates may appear, so we check again then.

In [ ]:
reviews_per_recipe = interactions.groupby("recipe_id").size()
recipes["n_reviews"] = recipes["id"].map(reviews_per_recipe).fillna(0).astype(int)

n_before = len(recipes)
recipes = (recipes.assign(ingr_set=ingr_set)
                  .sort_values("n_reviews", ascending=False)
                  .drop_duplicates("ingr_set")
                  .drop(columns="ingr_set")
                  .sort_index()
                  .reset_index(drop=True))
print(f"Removed {n_before - len(recipes):,} duplicate recipes, {len(recipes):,} left")

n_before = len(interactions)
interactions = interactions[interactions["recipe_id"].isin(recipes["id"])].reset_index(drop=True)
print(f"Removed {n_before - len(interactions):,} reviews of the removed copies, {len(interactions):,} left")

### Duplicate reviews

No user reviewed the same recipe twice (step 1). But some users wrote exactly the same text on different recipes:

In [ ]:
same_text = interactions[interactions.duplicated(["user_id", "review"], keep=False)]
print("Reviews with the same text by the same user:", len(same_text))
same_text["review"].value_counts().head(8)

These are short, generic reviews like *Yum!* and *Delicious!*. They are real reviews of different recipes, not copies, so we keep them.

## Step 6: Choose which recipes become nodes

The course page (week 8) says *"A few hundred to a few thousand nodes is the sweet spot"* and that above 10,000 nodes the project becomes about managing compute. It also asks for *text on the nodes*. So we keep recipes that have many reviews: they are recipes people actually cook, and each has enough review text.

First we see how many recipes we get with different minimum numbers of reviews.

In [ ]:
reviews_per_recipe = interactions.groupby("recipe_id").size()

rows = []
for k in [10, 20, 30, 50, 100]:
    kept = reviews_per_recipe[reviews_per_recipe >= k]
    rows.append({"min_reviews": k,
                 "recipes": len(kept),
                 "share_of_recipes": len(kept) / len(recipes),
                 "share_of_reviews": kept.sum() / len(interactions)})

pd.DataFrame(rows).round(3)

We choose **at least 30 reviews**: about 4,500 recipes is in the course's sweet spot, and every recipe has enough review text to analyse.

In [ ]:
MIN_REVIEWS = 30

all_recipes = recipes   # keep the full set, to compare with below
recipes = recipes[recipes["n_reviews"] >= MIN_REVIEWS].reset_index(drop=True)
interactions = interactions[interactions["recipe_id"].isin(recipes["id"])].reset_index(drop=True)

print(f"Recipes: {len(recipes):,}")
print(f"Reviews: {len(interactions):,}")
print(f"Users:   {interactions['user_id'].nunique():,}")

**Limitation:** we now study *popular* recipes, not all recipes on Food.com. To see how they differ, we compare them with all recipes:

In [ ]:
all_recipes["year"] = all_recipes["submitted"].str[:4].astype(int)
recipes["year"] = recipes["submitted"].str[:4].astype(int)

cols = ["n_ingredients", "n_steps", "minutes", "year"]
pd.DataFrame({"all recipes": all_recipes[cols].median(),
              "kept (>= 30 reviews)": recipes[cols].median()})

The kept recipes are a bit simpler (8 vs 9 ingredients and steps), take the same time, but were submitted about **four years earlier** (median 2003 vs 2007).

**Note: rich get richer.** This fits the idea of preferential attachment from week 2 (Barabási–Albert): a recipe that already has many reviews is shown higher on the site and in searches, so it is found and reviewed by even more people. Early recipes have had the longest time to collect reviews, and every lead they get is reinforced, just like the oldest nodes become hubs in the BA model.

But we have not shown that this mechanism is at work. The age difference is also explained by time alone: an older recipe has simply been online longer. To separate the two, we would have to look at how fast recipes collect reviews over time. In the report we can therefore only say that the age difference is *consistent with* rich get richer, and that our sample is biased towards old, popular recipes.

## Step 7: Ingredient names

The links in our network will be based on shared ingredients, so two recipes that both use eggs must also *write* eggs the same way. First we look at how the names look, before we change anything.

In [ ]:
from collections import Counter

ingredient_counts = Counter(x for xs in recipes["ingredients"] for x in xs)
print("Different ingredient names:", len(ingredient_counts))
print("Names used in only one recipe:", sum(c == 1 for c in ingredient_counts.values()))

ingredient_counts.most_common(30)

Variants of the same ingredient, e.g. everything that contains *egg* or *garlic*:

In [ ]:
for word in ["egg", "garlic", "butter", "pepper"]:
    variants = {name: c for name, c in ingredient_counts.items() if word in name}
    print(word, "->", len(variants), "names:", sorted(variants, key=variants.get, reverse=True)[:12])
    print()

### The plan

Ingredient names are short texts, so we clean them with the preprocessing ideas from week 5. Every step can also remove useful information, so we do one step at a time and count how many names are left after each step.

1. **Remove describing words** (*fresh, minced, chopped, large, unsalted, ...*), like a stopword list, plus brand names (*lipton, swanson, ...*) and sizes (*8-inch*, *95%*). We only remove whole words, so *peanut butter* and *eggplant* are never touched. A few names would change meaning if a word were removed (*dry mustard* is mustard powder, not mustard), so they are fixed first (`KEEP`). *lemon, juice of* is rewritten to *lemon juice*.
2. **Plural to singular** (*eggs → egg*, *tomatoes → tomato*, *leaves → leaf*), a simple form of lemmatization. We only change a word if the singular form also exists as an ingredient name in the data, so *molasses* stays *molasses*. We do this after step 1, because *fresh garlic cloves* first has to become *garlic cloves* before it can become *garlic clove*.
3. **A hand-made list of synonyms** (*garlic clove → garlic*, *whipping cream → heavy cream*), splitting combined ingredients (*salt and pepper → salt* + *pepper*), leaves of herbs count as the herb (*thyme leaf → thyme*, but *bay leaf* stays), and **removing things that are not food** (aluminum foil, toothpicks, skewers).

The rules were made in two rounds: a first version, and then a second round after we read through all the clean names and the hand-check samples (see below). Many synonyms and the `KEEP` names come from that second round.

In [ ]:
import re

# Step 1: names where removing a word would change the ingredient, so they are fixed first
KEEP = {
    "dry mustard": "mustard powder", "dried mustard": "mustard powder", "ground mustard": "mustard powder",
    "dry milk": "milk powder", "powdered milk": "milk powder", "nonfat dry milk": "milk powder",
    "lean ground meat": "ground meat", "ground meat": "ground meat",
    "1% fat cottage cheese": "cottage cheese",
}

# Step 1: describing words, brand names and phrases that do not change what the ingredient is
PHRASES = ["extra virgin", "extra-virgin", "low sodium", "reduced sodium", "low fat", "fat free",
           "reduced fat", "all-purpose", "all purpose", "freshly ground", "fresh ground",
           "old fashioned", "old-fashioned", "whole grain", "whole-grain", "head of", "ear of", "ears of",
           # brand names (not kept where the brand is the product: velveeta, cool whip, bisquick, jell-o)
           "campbell's", "kraft", "nestle", "toll house", "pillsbury", "hidden valley", "jimmy dean",
           "hellmann's", "hellmanns", "philadelphia", "heinz", "lea & perrins", "betty crocker", "hormel",
           "lipton", "knorr", "mccormick's", "mccormick", "kikkoman's", "kikkoman", "swanson", "old el paso"]
WORDS = {"fresh", "freshly", "minced", "chopped", "diced", "sliced", "grated", "shredded", "crushed",
         "large", "small", "medium", "unsalted", "salted", "ground", "finely", "coarsely", "boneless",
         "skinless", "lean", "extra", "virgin", "granulated", "kosher", "plain", "pure", "cooked",
         "uncooked", "frozen", "canned", "softened", "melted", "beaten", "peeled", "pitted", "boiling",
         "cold", "warm", "lukewarm", "packed", "light", "dark", "sharp", "mild", "organic", "baby",
         "low-fat", "fat-free", "nonfat", "non-fat", "reduced-fat", "low-sodium", "reduced-sodium",
         "whole", "raw", "dried", "dry",
         "semi-sweet", "semisweet", "unsweetened", "miniature", "mini", "rolled", "quick-cooking", "quick",
         "creamy", "chunky", "crunchy", "smooth", "slivered", "coarse", "cracked", "unbleached", "bleached",
         "wide", "fillet", "fillets", "original", "jumbo"}


# sizes and percentages like 8-inch, 9", 95%, 10-minute (but not 7-up, v8, a.1.)
SIZE = re.compile(r'^\d+(%|"|-inch|-minute)$')


def remove_describing_words(name):
    if name in KEEP:
        return KEEP[name]
    # "lemon, juice of" -> "lemon juice", "limes, zest of" -> "lime zest"
    m = re.match(r"^(\w+?)s?, (juice|zest|rind|juice and zest|juice and rind) of$", name)
    if m:
        fruit, part = m.groups()
        parts = {"juice": "juice", "zest": "zest", "rind": "zest"}
        return "; ".join(f"{fruit} {parts[p]}" for p in part.split(" and "))
    for phrase in PHRASES:
        name = re.sub(rf"\b{re.escape(phrase)}(?!\w)", " ", name)
    words = [w for w in name.split() if w not in WORDS and not SIZE.match(w)]
    return " ".join(words) if words else name      # never remove everything


# Step 2: plural -> singular, only if the singular form exists in the data
def to_singular(name, known_names):
    if name.endswith("'s"):                        # m&m's, not a plural
        return name
    if name.endswith("leaves"):                    # thyme leaves -> thyme leaf
        return name[:-6] + "leaf"
    for plural, singular in [("ies", "y"), ("oes", "o"), ("ches", "ch"), ("ves", "f"), ("s", "")]:
        if name.endswith(plural) and not name.endswith("ss"):
            candidate = name[: -len(plural)] + singular
            if candidate in known_names:
                return candidate
    return name


# Step 3: synonyms; "; " splits one name into two ingredients
SYNONYMS = {
    "garlic clove": "garlic",
    "clove garlic": "garlic",
    "black pepper": "pepper",
    "salt and pepper": "salt; pepper",
    "salt & pepper": "salt; pepper",
    "salt and black pepper": "salt; pepper",
    "salt & black pepper": "salt; pepper",
    "sea salt": "salt",
    "table salt": "salt",
    "white sugar": "sugar",
    "confectioners' sugar": "powdered sugar",
    "icing sugar": "powdered sugar",
    "vanilla extract": "vanilla",
    "oil": "vegetable oil",
    "cooking oil": "vegetable oil",
    "salad oil": "vegetable oil",
    "scallion": "green onion",
    "spring onion": "green onion",
    "catsup": "ketchup",
    "chicken breast half": "chicken breast",
    # found when we read through the clean names
    "cayenne": "cayenne pepper",
    "jalapeno pepper": "jalapeno",
    "celery rib": "celery",
    "celery stalk": "celery",
    "dill weed": "dill",
    "basil leaf": "basil",
    "oregano leaf": "oregano",
    "oregano leaves": "oregano",
    "parsley leaf": "parsley",
    "parsley leaves": "parsley",
    "parsley flakes": "parsley",
    "spinach leaf": "spinach",
    "spinach leaves": "spinach",
    "cocoa powder": "cocoa",
    "whipping cream": "heavy cream",
    "heavy whipping cream": "heavy cream",
    "half-and-half cream": "half-and-half",
    "condensed cream of mushroom soup": "cream of mushroom soup",
    "condensed cream of chicken soup": "cream of chicken soup",
    "condensed cream of celery soup": "cream of celery soup",
    "condensed cream of tomato soup": "cream of tomato soup",
    "elbow macaroni": "macaroni",
    "feta": "feta cheese",
    "cheddar": "cheddar cheese",
    "parmesan": "parmesan cheese",
    "mozzarella": "mozzarella cheese",
    "taco seasoning mix": "taco seasoning",
    "hot water": "water",
    "ice water": "water",
    "kernel corn": "corn",
    "corn kernel": "corn",
    "apple cider vinegar": "cider vinegar",
    "chicken bouillon cube": "chicken bouillon",
    "chicken bouillon granules": "chicken bouillon",
    "beef bouillon cube": "beef bouillon",
    "beef bouillon granules": "beef bouillon",
    "bicarbonate of soda": "baking soda",
    "italian seasoned breadcrumbs": "italian breadcrumbs",
    "italian style breadcrumbs": "italian breadcrumbs",
    "pecan half": "pecans",
    "pecan halves": "pecans",
    "rotel": "rotel tomatoes",
    "ro-tel tomatoes": "rotel tomatoes",
    "chocolate chip": "chocolate chips",
    "powdered milk": "milk powder",
    "grain mustard": "mustard",
    "gingerroot": "ginger",
    "milk ricotta cheese": "ricotta cheese",
}

# Step 3: leaves of herbs count as the herb (thyme leaf -> thyme), except leaves that are an ingredient themselves
LEAF_NAMES = {"bay leaf", "curry leaf", "banana leaf", "kaffir lime leaf", "lime leaf", "grape leaf"}

# Not food: kitchen equipment that some recipes list as an ingredient
NOT_FOOD = {"aluminum foil", "reynolds wrap foil", "wooden skewer", "bamboo skewer", "toothpick", "paraffin wax",
            "parchment paper", "wax paper", "plastic wrap", "oven cooking bag", "jar", "tall slender 4 cup container"}

Apply the three steps to every different raw name, and count how many different names are left after each step.

In [ ]:
raw_names = set(x for xs in recipes["ingredients"] for x in xs)

# names known in the full data (all recipes), so more singular forms can be found
known_names = set(x for xs in all_recipes["ingredients"] for x in xs)
known_names |= set(remove_describing_words(x) for x in known_names)


def clean_name(name):
    result = []
    for part in remove_describing_words(name).split("; "):    # step 1
        part = to_singular(part, known_names)                  # step 2
        for p in SYNONYMS.get(part, part).split("; "):         # step 3 (one name can become two)
            if p.endswith(" leaf") and p not in LEAF_NAMES:
                p = p[:-5]
            if p not in NOT_FOOD:
                result.append(p)
    return result


names1 = set(part for x in raw_names for part in remove_describing_words(x).split("; "))
names2 = set(to_singular(x, known_names) for x in names1)
names3 = set(part for x in raw_names for part in clean_name(x))
print(f"Raw names:              {len(raw_names):,}")
print(f"1. describing words:    {len(names1):,}")
print(f"2. plural -> singular:  {len(names2):,}")
print(f"3. synonyms, not food:  {len(names3):,}")

In [ ]:
clean = {x: clean_name(x) for x in raw_names}
recipes["ingr_clean"] = recipes["ingredients"].apply(
    lambda xs: sorted(set(part for x in xs for part in clean[x])))   # each ingredient once per recipe

Counter(recipes["ingr_clean"].explode()).most_common(30)

**Names used in only one recipe** (hapax, week 5). Such a name can never create a link, because a link needs two recipes that share an ingredient. How many are there before and after cleaning, and how many of the raw ones got a partner?

In [ ]:
raw_counts = Counter(x for xs in recipes["ingredients"] for x in set(xs))
clean_counts = Counter(x for xs in recipes["ingr_clean"] for x in xs)

raw_alone = [x for x, c in raw_counts.items() if c == 1]
got_partner = [x for x in raw_alone if any(clean_counts[part] > 1 for part in clean[x])]

print(f"Before cleaning: {len(raw_alone):,} of {len(raw_counts):,} names are used in only one recipe")
print(f"After cleaning:  {sum(c == 1 for c in clean_counts.values()):,} of {len(clean_counts):,} names are used in only one recipe")
print(f"Raw names that were alone but now share a clean name with another recipe: {len(got_partner):,}")
print("Examples:", [(x, clean[x]) for x in got_partner[:8]])

### Check 1: read the changes by hand

A random sample of 40 raw names that were changed. We read them and count how many are wrong.

In [ ]:
changes = pd.DataFrame({"raw": list(clean), "clean": ["; ".join(v) for v in clean.values()]})
changes = changes[changes["raw"] != changes["clean"]]
print("Raw names that were changed:", len(changes))
changes.sample(40, random_state=0).sort_values("clean")

Result of our reading of the hand-check samples:

We read a sample of 40 changes after each version of the rules, and fixed what was wrong. Errors we found and fixed this way:

- *dry thyme leaves → thyme leave* (plural rule) → now *thyme*
- *lean ground meat → meat* (too vague) → now *ground meat*
- *dry mustard → mustard* (mustard powder is not mustard) → now *mustard powder*
- *a.1. original sauce → original sauce*, *7-up soda → soda*, *spicy hot v8 → spicy hot* (the rule for numbers was too broad) → now only sizes like *8-inch* and *95%* are removed
- *dried curry leaves → curry* → curry leaves are kept
- *cooked whole grain brown rice → grain brown rice*, *whole milk ricotta cheese → milk ricotta cheese*

In the **final sample** (the table above) **0 of 40 changes are wrong**. One merge is missed: *hass avocado* and *avocado* are still two names.

Because we fixed the rules after reading samples, the final sample is the only one that is a fair test of the final rules. **Please read the sample yourself as well**: a hand check is only a check if a person has done it.

### Check 2: compare with a second method

The authors of the dataset made their own cleaning of the ingredient names (`ingr_map.pkl`, Majumder et al. 2019), which maps each raw name to a clean name. Their cleaning has its own errors, e.g. *all-purpose flour* became *all-purpose flmy* and *asparagus* became *asparagu*, so it is a comparison, not the truth.

We count **merges**: pairs of raw names that end up with the same clean name. How many of their merges do we also make, and how many of ours do they also make?

(We tried NMI from week 4 first, but it is not useful here: most names are alone in their group, so even a shuffled grouping gets NMI ≈ 0.94.)

In [ ]:
from itertools import combinations

ingr_map = pd.read_pickle(os.path.join(path, "ingr_map.pkl"))
theirs = dict(zip(ingr_map["raw_ingr"], ingr_map["replaced"]))
both = [x for x in raw_names if x in theirs]
print(f"Raw names that are also in their map: {len(both):,} of {len(raw_names):,}")


def merged_pairs(labels):
    groups = pd.Series(both).groupby(labels)
    return set(pair for g in groups for pair in combinations(sorted(g[1]), 2))


compare = pd.DataFrame({"raw": both, "ours": ["; ".join(clean[x]) for x in both], "theirs": [theirs[x] for x in both]})
ours_pairs = merged_pairs(compare["ours"].values)
theirs_pairs = merged_pairs(compare["theirs"].values)
shared = ours_pairs & theirs_pairs
print(f"Our merges: {len(ours_pairs):,}, their merges: {len(theirs_pairs):,}, made by both: {len(shared):,}")
print(f"Share of their merges that we also make: {len(shared) / len(theirs_pairs):.0%}")
print(f"Share of our merges that they also make: {len(shared) / len(ours_pairs):.0%}")

Where do we disagree? Names that *we* merge but *they* keep apart, and the other way around:

In [ ]:
we_merge = compare.groupby("ours")["theirs"].nunique()
they_merge = compare.groupby("theirs")["ours"].nunique()
print("Our names that they split into several:", (we_merge > 1).sum())
print("Their names that we split into several:", (they_merge > 1).sum())

compare[compare["ours"].isin(we_merge[we_merge > 1].index)].sort_values("ours").head(20)

In [ ]:
compare[compare["theirs"].isin(they_merge[they_merge > 1].index)].sort_values("theirs").head(20)

The two methods make different *kinds* of merges, so the overlap is low:

- **They merge by type of food**: all 34 kinds of sausage become *sausage*, all dressings become *dressing*, and both *tortilla chips* and *flour tortillas* become *tortilla*. This gives many pairs (one group of 34 names alone gives 561 pairs) but also errors.
- **We only merge versions of the same ingredient**: fresh and frozen shrimp, cooked and raw rice, dried and fresh dill.

So *when two ingredients are the same* is a choice, and we state ours in the report. The comparison also showed something we missed at first: sizes and percentages (*8-inch flour tortillas*, *95% lean ground beef*), which we now remove in step 1.

#### Could we just use their mapping instead?

We try it: clean our recipes with their mapping and compare with ours.

In [ ]:
theirs_clean = recipes["ingredients"].apply(lambda xs: sorted(set(theirs.get(x, x) for x in xs)))

broken = [name for name in set(theirs.values()) if any(w in name for w in ["flmy", "pastum", "asparagu"])]
pp_recipes = pd.read_csv(os.path.join(path, "PP_recipes.csv"), usecols=["id"])

print("Different names, ours:  ", recipes["ingr_clean"].explode().nunique())
print("Different names, theirs:", theirs_clean.explode().nunique())
print("Our raw names missing in their map:", len(raw_names) - len(both))
print("Broken names in their map (e.g. flmy, pastum):", len(broken), "e.g.", sorted(broken)[:5])
print("Our recipes in their preprocessed file PP_recipes.csv:", recipes["id"].isin(pp_recipes["id"]).sum(), "of", len(recipes))

We keep **our cleaning as the main method**, because their mapping:

- has broken names (*flour → flmy*, *pasta → pastum*). They are consistent, so merging still works, but they break the climate matching later, where *flour* must be recognised as wheat;
- misses some of our raw names;
- merges different things (*tortilla chips* and *flour tortillas* both become *tortilla*);
- and their preprocessed file `PP_recipes.csv` does not cover all our recipes, so using it would cost us nodes.

**Robustness check for later:** when the network is built, we build it again with their mapping and check whether the communities and the main results stay the same. That is a stronger check than comparing names, because it shows whether this choice changes our conclusions.

### Check 3: duplicates again

After cleaning, two recipes that wrote the same ingredients differently can now have exactly the same list.

In [ ]:
clean_set = recipes["ingr_clean"].apply(tuple)
print("Recipes with the same clean ingredients as another recipe:", clean_set.duplicated().sum())
recipes[clean_set.duplicated(keep=False)].assign(s=clean_set).sort_values("s")[["name", "ingredients", "n_reviews"]].head(10)

These 45 are **different recipes** that happen to use the same basic ingredients, e.g. bread-machine rolls and crusty bread (milk, egg, butter, sugar, salt, flour, yeast). They have different names and many reviews each, so they are not re-uploads, and we keep them. They show a limitation of our data: without amounts, a bun and a loaf of bread look the same.

### Summary of step 7

- 3,111 raw names → 2,129 clean names, stored in `recipes["ingr_clean"]` (each ingredient once per recipe).
- 413 names that were used in only one recipe now share a name with another recipe, so they can create links.
- Hand check: 0 of 40 changes wrong in the final sample, after fixing the errors found in earlier samples.
- Second method: the dataset's own cleaning merges by *type of food*, we merge *versions of the same ingredient*, so the overlap is low (26 % / 28 %). This is a choice we state in the report.
- 45 recipes now share their ingredient list with another recipe; they are kept.

## Overview of the cleaned data

In [ ]:
recipes["n_ingr_clean"] = recipes["ingr_clean"].str.len()

print(f"Recipes (nodes):         {len(recipes):,}")
print(f"Reviews:                 {len(interactions):,}")
print(f"Users:                   {interactions['user_id'].nunique():,}")
print(f"Different ingredients:   {recipes['ingr_clean'].explode().nunique():,}")
print(f"Submitted:               {recipes['year'].min()}-{recipes['year'].max()}")
print(f"Reviews written:         {interactions['date'].min()} to {interactions['date'].max()}")

In [ ]:
recipes[["n_reviews", "n_ingr_clean", "n_steps", "minutes", "year"]].describe().round(1)

In [ ]:
recipes[["id", "name", "n_reviews", "ingr_clean"]].head(10)

In [ ]:
interactions["rating"].value_counts(dropna=False).sort_index()

## Degree distribution of the ingredients

If we see the data as a bipartite network (recipes linked to the ingredients they use), the **degree of an ingredient** is the number of recipes it is used in. Salt is a hub, and many ingredients have degree 1.

Left: how many ingredients have each degree, on log–log axes (week 1). Right: the CCDF, the share of ingredients with degree at least *k* (week 2). It uses every data point and needs no binning.

In [ ]:
import matplotlib.pyplot as plt

ingredient_degree = np.array(list(clean_counts.values()))
print(f"Ingredients: {len(ingredient_degree):,}, degree 1: {(ingredient_degree == 1).sum():,}, "
      f"median: {np.median(ingredient_degree):.0f}, max: {ingredient_degree.max():,} ({clean_counts.most_common(1)[0][0]})")

counts = np.bincount(ingredient_degree)
k = np.arange(len(counts))
ccdf = [(ingredient_degree >= x).mean() for x in k[1:]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.loglog(k[counts > 0], counts[counts > 0], ".")
ax1.set_xlabel("Degree $k$ (number of recipes using the ingredient)")
ax1.set_ylabel("Number of ingredients")
ax1.set_title("Degree distribution")

ax2.loglog(k[1:], ccdf, ".")
ax2.set_xlabel("Degree $k$")
ax2.set_ylabel("Share of ingredients with degree $\\geq k$")
ax2.set_title("CCDF")

plt.tight_layout()
plt.show()

**What we see:** the distribution is heavy-tailed. 893 of 2,129 ingredients (42 %) are used in only one recipe, the median is 2, but a few hubs are used in thousands of recipes (salt in 2,536 of 4,467). Left: the tail is noisy, because most degrees above ~40 have only 0 or 1 ingredient. Right: the CCDF is smooth and close to a straight line over about three decades.

**What we can and cannot say (week 2):** a straight-ish line on log–log axes shows a heavy tail, not that it is a power law with a certain exponent. That would need a proper fit (maximum likelihood) and a comparison with other distributions, e.g. lognormal. It is the same pattern as word frequencies (Zipf, week 5): ingredient names behave like words.

## Save the cleaned data

Other notebooks (e.g. `reviews.ipynb`) load the cleaned data from these files, so the preprocessing above only has to run once. The folder `data/` is not on GitHub (see `.gitignore`), so run this notebook once to create it.

In [ ]:
os.makedirs("data", exist_ok=True)

recipes.to_parquet("data/recipes_clean.parquet")
interactions.to_parquet("data/reviews_clean.parquet")
print(f"Saved {len(recipes):,} recipes and {len(interactions):,} reviews to data/")

## Step 8: Define the links

Two recipes are linked if they use similar ingredients. We build this in four steps, each from the course.

### 1. Each recipe as a vector of ingredients (week 5, Bag of Words)

A recipe × ingredient matrix with a 1 where the recipe uses the ingredient.

We keep **all** ingredients, also the ones used in only one recipe. They cannot create a link, but they belong to the recipe: a prime rib roast whose main ingredient (prime rib) is used nowhere else should *not* look identical to a seasoning mix just because they share garlic powder, pepper and salt. We first tried to leave them out, and then exactly that happened (cosine 1.00).

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
X = mlb.fit_transform(recipes["ingr_clean"])
print("Matrix:", X.shape, "(recipes x ingredients)")
print(f"Share of zeros: {(X == 0).mean():.1%}")

### 2. Weight the ingredients with TF-IDF (week 6)

idf(t) = ln(N / df(t)), where df(t) is the number of recipes that use ingredient t. An ingredient in every recipe gets weight 0, a rare one gets a high weight. So sharing salt counts for almost nothing, and sharing a rare ingredient counts for a lot.

TF is 1 or 0 here, because each ingredient is listed once per recipe. (Dividing by the number of ingredients, as on the course page, would not change anything, because cosine ignores the length of the vectors.)

In [ ]:
N = len(recipes)
df = X.sum(axis=0)
idf = np.log(N / df)
tfidf = X * idf

idf_table = pd.Series(idf, index=mlb.classes_).sort_values()
print("Lowest idf (common):", idf_table.head(5).round(2).to_dict())
print("Highest idf (rare):", idf_table.tail(3).round(2).to_dict())

### 3. Similarity between recipes with cosine (week 6)

cos(A, B) = A·B / (‖A‖‖B‖). We divide every row by its length, so the similarity of all pairs at once is the matrix product. The cosine is between 0 (nothing in common) and 1 (the same ingredients).

In [ ]:
norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
unit = np.divide(tfidf, norms, out=np.zeros_like(tfidf), where=norms > 0)
S = unit @ unit.T
np.fill_diagonal(S, 0)   # no links from a recipe to itself

pairs = S[np.triu_indices(N, k=1)]
print(f"Pairs of recipes: {len(pairs):,}")
print(f"Pairs with similarity > 0: {(pairs > 0).sum():,} ({(pairs > 0).mean():.0%})")
print(f"Median similarity of those: {np.median(pairs[pairs > 0]):.3f}")

Almost all pairs share at least one ingredient, so the full network would be almost complete: a hairball where every recipe is linked to most others. We need to keep only the links that matter.

### 4. Keep the important links with the disparity filter (week 4)

A global threshold (e.g. keep all links above 0.3) would remove all links of recipes whose similarities are generally low. The disparity filter instead tests every link against the recipe's own links: for recipe i with strength s_i (sum of its weights) and degree k_i,

α_ij = (1 − w_ij / s_i)^(k_i − 1)

The link is kept if α_ij < α for at least one of its two ends. So every recipe keeps the links that are unusually strong *for that recipe*.

In [ ]:
k = (S > 0).sum(axis=1)   # degree in the full network
s = S.sum(axis=1)         # strength in the full network

with np.errstate(divide="ignore", invalid="ignore"):
    alpha_ij = (1 - S / s[:, None]) ** (k[:, None] - 1)   # seen from recipe i (rows)


def backbone(alpha):
    keep = (S > 0) & ((alpha_ij < alpha) | (alpha_ij.T < alpha))
    i, j = np.where(np.triu(keep, k=1))
    G = nx.Graph()
    G.add_nodes_from(recipes["id"])            # all nodes first (week 1), so isolated recipes are kept
    ids = recipes["id"].to_numpy()
    G.add_weighted_edges_from(zip(ids[i], ids[j], S[i, j]))
    return G

How does the network change with α? Lower α = stricter = fewer links.

Note: the full network is very dense (each recipe has about 2,700 neighbours), and then (1 − w/s)^(k−1) becomes very small as soon as a link is just a few times stronger than the recipe's average. So α must be much lower here than for the sparse philosophers network in week 4.

In [ ]:
import networkx as nx

rows = []
for a in [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 0.05]:
    B = backbone(a)
    w = np.array([d for _, _, d in B.edges(data="weight")])
    giant = max(nx.connected_components(B), key=len)
    rows.append({"alpha": f"{a:g}", "links": B.number_of_edges(),
                 "avg_degree": 2 * B.number_of_edges() / N,
                 "isolated": nx.number_of_isolates(B),
                 "giant_component": len(giant) / N,
                 "median_cosine": np.median(w), "min_cosine": w.min()})
pd.DataFrame(rows).round(3)

We choose **α = 0.0001**:

- an average degree of about 11, close to the Marvel network from the course (9.5): sparse enough to see structure, not a hairball;
- 96 % of the recipes are in the giant component, and 160 are isolated;
- the kept links are clearly stronger than the typical pair (median cosine 0.38 vs 0.025 in the full network).

With α = 0.001 the network is almost three times denser, and with α = 0.00001 18 % of the recipes fall out of the giant component. We report this table, so the choice can be checked.

In [ ]:
ALPHA = 1e-4
G = backbone(ALPHA)
nx.set_node_attributes(G, recipes.set_index("id")["name"].to_dict(), "name")
nx.set_node_attributes(G, recipes.set_index("id")["n_reviews"].to_dict(), "n_reviews")

giant = max(nx.connected_components(G), key=len)
print(f"Nodes:            {G.number_of_nodes():,}")
print(f"Links:            {G.number_of_edges():,}")
print(f"Average degree:   {2 * G.number_of_edges() / G.number_of_nodes():.1f}")
print(f"Density:          {nx.density(G):.4f}")
print(f"Isolated recipes: {nx.number_of_isolates(G):,}")
print(f"Giant component:  {len(giant):,} ({len(giant) / G.number_of_nodes():.1%})")

### Check: read the links

Do the links make sense? We read the strongest links and a random sample, with the ingredients the two recipes share.

In [ ]:
ingr_of = dict(zip(recipes["id"], recipes["ingr_clean"]))
name_of = dict(zip(recipes["id"], recipes["name"]))


def show(edges):
    for u, v, w in edges:
        shared = sorted(set(ingr_of[u]) & set(ingr_of[v]))
        print(f"{w:.2f}  {name_of[u][:35]:35s} | {name_of[v][:35]:35s} | {', '.join(shared)}")


print("Strongest links:")
show(sorted(G.edges(data="weight"), key=lambda e: -e[2])[:10])

In [ ]:
import random

random.seed(0)
print("Random links:")
show(random.sample(list(G.edges(data="weight")), 15))

**Result of our reading:**

- The strongest links (cosine 1.00) are recipes with exactly the same clean ingredients, e.g. two buttercream frostings or two bread-machine rolls (the 45 recipes from check 3 in step 7).
- In the random sample, 12 of 15 links are clearly meaningful: two potato salads, two kinds of mashed potatoes, two peanut butter cookies, two whole-wheat pizza doughs, BBQ brisket and ribs.
- 3 of 15 are weak: they share one characteristic ingredient plus salt or water (rice pilaf with peas and tuna pasta salad; pancakes and maple-glazed chicken; a cream sauce and creamy leeks). The disparity filter keeps every recipe's *relatively* strongest links, so a recipe with few or unusual ingredients can get links that are weak in absolute terms.

So a link means *the two recipes share ingredients that are unusual for them*, and most often that they are the same type of dish.

### What is a backbone?

A **backbone** is a thinned-out version of a network that keeps only the most important links, so the basic structure becomes visible.

Our full network links two recipes if they share just one ingredient: 6.1 million links, about 2,700 neighbours per recipe, most of them because of salt or butter. Drawn as it is, it would be a hairball. The disparity filter keeps, for every recipe, only the links that are unusually strong *for that recipe*: 25,644 links, 11.5 neighbours on average.

The course page (week 4) says: *"The backbone isn't data (it's intended for visualization)"*. So we use the backbone to **see** the structure (the layout below), and the full network to **measure** it (the communities below).

### Drawing the network

We follow the five steps from week 4 ("Some thoughts on network visualization"):

1. **Backbone first**: the disparity filter above.
2. **Layout of the backbone**, not of the full network: a force layout (Fruchterman–Reingold, computed with igraph because it is much faster than NetworkX).
3. **Colour by community**, computed on the **full** weighted network (all pairs with cosine > 0), not on the backbone, because the course page says: "the backbone isn't data (it's intended for visualization)". We use Louvain (week 4). Each community is named after its most typical ingredients (highest mean TF-IDF), and the colours are the Okabe–Ito palette, which colour-blind readers can tell apart.
4. **Size by degree** in the backbone.
5. **Label the most important nodes** (the two with most links in each community), and a caption that says the filter, the parameter and what was removed.

The communities are *not tested yet* (null model, stability across seeds); here they only help us read the drawing.

In [ ]:
import igraph as ig
import random

# Louvain on the full weighted network
i, j = np.where(np.triu(S, k=1) > 0)
g_full = ig.Graph(n=N, edges=list(zip(i, j)), edge_attrs={"weight": S[i, j]})
random.seed(42)
louvain = g_full.community_multilevel(weights="weight")
community = dict(zip(recipes["id"], louvain.membership))
print(f"Louvain on the full network: {len(louvain)} communities, modularity {louvain.modularity:.2f}")

# Name each community after the ingredients with the highest mean TF-IDF
labels = {}
for c in range(len(louvain)):
    rows = np.array(louvain.membership) == c
    top = pd.Series(tfidf[rows].mean(axis=0), index=mlb.classes_).nlargest(4).index
    labels[c] = ", ".join(top)
    print(f"Community {c}: {rows.sum():,} recipes: {labels[c]}")

In [ ]:
# Force layout of the backbone's giant component
Gc = G.subgraph(giant)
nodes = list(Gc)
g_back = ig.Graph.from_networkx(Gc)
layout = g_back.layout_fruchterman_reingold(weights=[w for _, _, w in Gc.edges(data="weight")], niter=1000, seed=None)
pos = dict(zip(nodes, layout.coords))

degree = dict(Gc.degree())
# label the 2 recipes with most links in each community, so the reader can find their way in every group
top = [n for c in range(len(louvain))
       for n in sorted([n for n in nodes if community[n] == c], key=degree.get, reverse=True)[:2]]
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]

fig, ax = plt.subplots(figsize=(12, 12))
nx.draw_networkx_edges(Gc, pos, ax=ax, width=0.2, alpha=0.1, edge_color="#555555")
nx.draw_networkx_nodes(Gc, pos, ax=ax, nodelist=nodes,
                       node_color=[palette[community[n] % len(palette)] for n in nodes],
                       node_size=[2 + degree[n] for n in nodes],
                       linewidths=0.2, edgecolors="white", alpha=0.9)
for n in top:
    ax.annotate(" ".join(name_of[n].split())[:30], pos[n], xytext=(0, 8), textcoords="offset points",
                fontsize=8.5, ha="center", bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#999999", lw=0.5, alpha=0.9))

handles = [plt.Line2D([], [], marker="o", ls="", markersize=9, color=palette[c % len(palette)],
                      label=f"{labels[c]}  ({louvain.membership.count(c):,})") for c in range(len(louvain))]
ax.legend(handles=handles, title="Community (most typical ingredients, size)", loc="lower left", frameon=False, fontsize=9)
ax.set_title("Food.com recipes linked by shared ingredients", fontsize=14)
ax.axis("off")
fig.text(0.5, 0.06,
         f"Giant component of the disparity-filter backbone (α = {ALPHA:g}): {Gc.number_of_nodes():,} of {N:,} recipes and "
         f"{Gc.number_of_edges():,} links ({nx.number_of_isolates(G)} isolated recipes and small components not shown).\n"
         "Links: cosine similarity of TF-IDF ingredient vectors. Colour: Louvain community on the full weighted network. "
         "Size: degree in the backbone. Labels: the two recipes with most links in each community.",
         ha="center", fontsize=8.5, color="#444444")
plt.show()

## Step 9: Communities

Louvain (week 4) finds the number of communities itself, by moving recipes between groups until the modularity Q cannot get higher. But the **resolution** γ decides how big the groups become: Q = Σ [ ℓ_c/m − γ (d_c/2m)² ]. A higher γ punishes big groups more, so it gives more and smaller communities.

With the default γ = 1 we get only 3 very broad groups (savoury, sweet, Asian-style). We use **γ = 1.5**, which splits them into types of dishes. We run Louvain on the **full** weighted network, not the backbone (week 4).

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

GAMMA = 1.5

random.seed(42)
part = g_full.community_multilevel(weights="weight", resolution=GAMMA)
Q_real = g_full.modularity(part.membership, weights="weight", resolution=GAMMA)
print(f"Louvain, gamma = {GAMMA}: {len(part)} communities, Q = {Q_real:.3f}")

for c in sorted(range(len(part)), key=lambda c: -len(part[c])):
    rows = np.array(part.membership) == c
    top = pd.Series(tfidf[rows].mean(axis=0), index=mlb.classes_).nlargest(5).index
    print(f"{rows.sum():5,} recipes: {', '.join(top)}")

Save the communities, so `reviews.ipynb` can compare them (both the γ = 1.5 communities and the 3 broad γ = 1 communities used in the drawing).

In [ ]:
pd.DataFrame({"id": recipes["id"],
              "community": part.membership,           # gamma = 1.5
              "community_broad": louvain.membership,  # gamma = 1
              }).to_parquet("data/communities.parquet")
print("Saved data/communities.parquet")

### Check 1: stability across seeds

Louvain visits the recipes in a random order, so two runs can give different communities (week 4). We run it 10 times with different seeds and compare every pair of runs with NMI (1 = identical, 0 = nothing in common).

In [ ]:
from itertools import combinations

runs = []
for seed in range(10):
    random.seed(seed)
    p = g_full.community_multilevel(weights="weight", resolution=GAMMA)
    runs.append(p.membership)
    print(f"seed {seed}: {len(p)} communities, Q = {g_full.modularity(p.membership, weights='weight', resolution=GAMMA):.3f}")

nmis = [normalized_mutual_info_score(a, b) for a, b in combinations(runs, 2)]
print(f"\nNMI between runs: mean {np.mean(nmis):.2f}, min {np.min(nmis):.2f}, max {np.max(nmis):.2f}")

### Check 2: null model

Even a random network gets Q > 0, because Louvain always finds *some* groups (week 4). So we compare with a null model.

Our network is built from ingredient lists, so the null model shuffles the **ingredients**: every recipe keeps its number of ingredients, and every ingredient keeps the number of recipes it is used in. This is the degree-preserving shuffle from week 2, done on the recipe–ingredient data. We use the *curveball* algorithm: pick two recipes, pool the ingredients that only one of them has, and deal them out again at random (each recipe gets back as many as it gave).

Then we build TF-IDF, cosine and Louvain exactly as for the real data, and compare Q. (This cell takes some minutes: 20 networks with about 6 million links each.) If the real Q is much higher, the communities are about *which ingredients go together*, not just about how common the ingredients are.

In [ ]:
def curveball(lists, n_trades, rng):
    lists = [set(x) for x in lists]
    for _ in range(n_trades):
        a, b = rng.choice(len(lists), 2, replace=False)
        only_a, only_b = lists[a] - lists[b], lists[b] - lists[a]
        pool = list(only_a | only_b)
        rng.shuffle(pool)
        new_a = set(pool[:len(only_a)])
        lists[a] = (lists[a] - only_a) | new_a
        lists[b] = (lists[b] - only_b) | (set(pool) - new_a)
    return [sorted(x) for x in lists]


def build_full_graph(ingredient_lists):
    Xn = mlb.transform(ingredient_lists)           # same columns as the real data
    T = Xn * np.log(N / Xn.sum(axis=0))            # df is the same as in the real data
    T = T / np.linalg.norm(T, axis=1, keepdims=True)
    Sn = T @ T.T
    a, b = np.where(np.triu(Sn, k=1) > 0)
    return ig.Graph(n=N, edges=np.column_stack([a, b]), edge_attrs={"weight": Sn[a, b]})


rng = np.random.default_rng(0)
Q_null = []
for r in range(20):
    shuffled = curveball(recipes["ingr_clean"], n_trades=10 * N, rng=rng)
    gn = build_full_graph(shuffled)
    random.seed(r)
    pn = gn.community_multilevel(weights="weight", resolution=GAMMA)
    Q_null.append(gn.modularity(pn.membership, weights="weight", resolution=GAMMA))

Q_null = np.array(Q_null)
z = (Q_real - Q_null.mean()) / Q_null.std()
print(f"Real Q:  {Q_real:.3f}")
print(f"Null Q:  {Q_null.mean():.3f} ± {Q_null.std():.3f}  (20 shuffles, min {Q_null.min():.3f}, max {Q_null.max():.3f})")
print(f"z-score: {z:.0f}")

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.hist(Q_null, bins=10, color="#999999", label="Shuffled ingredients (20 networks)")
plt.axvline(Q_real, color="#D55E00", lw=2, label=f"Real network (Q = {Q_real:.2f})")
plt.xlabel(f"Modularity Q (Louvain, γ = {GAMMA})")
plt.ylabel("Number of networks")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

### What the checks show

**The communities are real structure (null model).** The real network has Q = 0.138, the 20 shuffled networks have Q = 0.061 ± 0.001 (z ≈ 100; the exact z changes a little between runs because the shuffles are random). So the shuffled networks also get Q > 0, as week 4 warns, but the real Q is more than twice as high. The groups come from *which ingredients are used together*, not only from how common the ingredients are.

**They are only moderately stable (seeds).** Between runs the number of communities goes from 12 to 16, Q hardly changes (0.135–0.138), and the NMI between runs is 0.70 on average (0.59–0.85). In the words of week 4: the core of the groups is stable, but the borders move. This is the flat modularity landscape: many different partitions have almost the same Q.

**What the 13 communities are** (our interpretation of the top ingredients, not the model's output): baking and sweets, Mediterranean/Italian (olive oil, parmesan), Mexican (cumin, chili powder, cilantro), yeast bread, cheese-and-bacon comfort food, Asian-style, spice rubs (garlic powder, paprika, cayenne), soups and stews (celery, carrot, potato, broth), American casseroles (ketchup, onion soup mix, cream of mushroom soup), plus some small groups. The smallest (5 recipes) is candy (m&m's, hershey kisses, pretzels).

**How to report it:** "Louvain (γ = 1.5) on the full network finds 12–16 communities across 10 seeds (NMI 0.59–0.85 between runs), with Q = 0.138 against 0.061 ± 0.001 for 20 ingredient-shuffled networks."